# Step 0 — Looking inside a digit classifier

The first project in a 6-step mechanistic interpretability curriculum. By the end of this notebook you'll have:

1. Trained your first neural network (logistic regression on MNIST).
2. Looked at the model's weights and seen that they encode something a human can recognise: fuzzy digit templates.
3. Trained a slightly larger model (a 1-hidden-layer MLP) and observed that its weights are messier — a teaser for the next project.

Read the `README.md` first if you haven't. It explains the concepts; this notebook is the runnable companion.

Expected runtime:.

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

torch.manual_seed(0)
np.random.seed(0)

## 2. Load MNIST

MNIST is 70,000 28×28 greyscale images of handwritten digits, each labelled 0-9. The first time you run this it'll download ~10MB; after that it's cached.

We flatten each image to a length-784 vector (28×28 = 784) and normalise pixel values to `[0, 1]`.

In [ ]:
train_data = datasets.MNIST('./data', train=True,  download=True)
test_data  = datasets.MNIST('./data', train=False, download=True)

train_x = train_data.data.float().reshape(-1, 784) / 255.0
train_y = train_data.targets
test_x  = test_data.data.float().reshape(-1, 784) / 255.0
test_y  = test_data.targets

train_x, train_y = train_x.to(device), train_y.to(device)
test_x,  test_y  = test_x.to(device),  test_y.to(device)

print(f'train: {train_x.shape}  labels: {train_y.shape}')
print(f'test:  {test_x.shape}  labels: {test_y.shape}')

# show a few examples
fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(train_x[i].cpu().reshape(28, 28), cmap='gray')
    ax.set_title(f'label: {train_y[i].item()}')
    ax.axis('off')
plt.suptitle('A few MNIST training examples')
plt.tight_layout()
plt.show()

## 3. Logistic regression — the simplest possible model

Just one linear layer: `784 → 10`. No hidden layer, no nonlinearity in the body. The model has shape `W: (10, 784)` plus a bias `b: (10,)` — 7,850 parameters total.

We train with Adam, using small mini-batches of 256 examples (random subsets of the training data). 5 epochs through the whole training set is plenty for this model to converge.

In [ ]:
class LogReg(nn.Module):
    def __init__(self):
        super().__init__()
        # nn.Linear is just a wrapper for W @ x + b
        self.linear = nn.Linear(784, 10)

    def forward(self, x):
        return self.linear(x)  # returns logits (raw class scores)

def train(model, train_x, train_y, n_epochs=5, batch_size=256, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    n_train = len(train_x)
    for epoch in range(n_epochs):
        perm = torch.randperm(n_train, device=train_x.device)
        running_loss = 0.0
        running_correct = 0
        for i in range(0, n_train, batch_size):
            idx = perm[i:i + batch_size]
            x, y = train_x[idx], train_y[idx]
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss    += loss.item() * len(x)
            running_correct += (logits.argmax(dim=1) == y).sum().item()
        print(f'epoch {epoch + 1}/{n_epochs}  train loss {running_loss / n_train:.4f}  train acc {running_correct / n_train:.3f}')

logreg = LogReg().to(device)
train(logreg, train_x, train_y, n_epochs=5)

with torch.no_grad():
    test_acc = (logreg(test_x).argmax(dim=1) == test_y).float().mean().item()
print(f'\nLogistic regression test accuracy: {test_acc:.3f}')

## 4. Visualise the weights as digit templates

Here's the wow moment. The model's weight matrix `W` has shape `(10, 784)` — one 784-dim row per digit class. If we reshape each row back to 28×28 and plot it as a heatmap, what do we see?

Hypothesis: since logistic regression scores each class by `W[c] · x + b[c]`, the model has effectively learnt a 28×28 "template" per class. Inputs that look like the template will get a high score for that class.

Let's check.

In [ ]:
W = logreg.linear.weight.detach().cpu().numpy()   # shape (10, 784)
vlim = np.abs(W).max()

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(W[i].reshape(28, 28), cmap='RdBu_r', vmin=-vlim, vmax=vlim)
    ax.set_title(f'class "{i}"')
    ax.axis('off')
plt.suptitle('Logistic regression weights — each row reshaped to 28×28', fontsize=13)
plt.tight_layout()
plt.show()

Look closely. Red regions are positive weights (the model wants ink there for this class); blue regions are negative (the model wants ink absent there for this class).

You should see:

- **"0"**: a red ring around the edge (where the loop of a 0 sits) with blue in the middle (since 0 has a hollow centre).
- **"1"**: a vertical red stripe down the middle.
- **"3", "8"**: stacked curves.
- **"7"**: a red horizontal bar on top and a diagonal stroke.

Each template is a blurry average of all the training examples of that digit, sculpted by training to discriminate between classes.

This is the whole algorithm. Logistic regression on MNIST is template matching. The model takes your input image, dots it against 10 stored templates, and predicts the class whose template gave the highest dot product. You can read the whole algorithm directly off the weight matrix. This is interpretability in its simplest form.

## 5. A 1-hidden-layer MLP

Now let's add one hidden layer of 32 neurons with a ReLU nonlinearity. This is a bigger model — `784 → 32 → 10`. Around 25,000 parameters.

We expect a small accuracy bump because the hidden layer can compute non-linear combinations of pixels (e.g. "this region is bright AND that region is dark").

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(784, n_hidden)
        self.fc2 = nn.Linear(n_hidden, 10)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

mlp = MLP(n_hidden=32).to(device)
train(mlp, train_x, train_y, n_epochs=5)

with torch.no_grad():
    test_acc = (mlp(test_x).argmax(dim=1) == test_y).float().mean().item()
print(f'\nMLP test accuracy: {test_acc:.3f}')

## 6. Visualise the MLP's first-layer weights

Each of the 32 hidden neurons has its own 784-dim input weight vector — a "filter" that determines what part of the input image makes this neuron fire. Let's plot all 32 filters as 28×28 heatmaps and see what they look like.

In [ ]:
W1 = mlp.fc1.weight.detach().cpu().numpy()   # shape (32, 784)
vlim = np.abs(W1).max()

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(W1[i].reshape(28, 28), cmap='RdBu_r', vmin=-vlim, vmax=vlim)
    ax.set_title(f'h{i}', fontsize=8)
    ax.axis('off')
plt.suptitle('MLP first-layer weights — 32 hidden neurons, each a 28×28 filter', fontsize=13)
plt.tight_layout()
plt.show()

Compare to Section 4. The logistic-regression templates were clean: one digit template per class, each visually recognisable.

The MLP's first-layer filters are messier. You'll likely see:

- Some filters that look like clean edges or strokes — parts of digits.
- Some that look like blurry blends of two or three digits at once.
- Some that look almost like random noise (or like a faint mix of many things).
- A few that look strikingly clean and others that don't.

Why are these messier? Because there are only 32 hidden neurons but a lot of useful features of digits a model could track — edges, curves, intersections, stroke directions, you name it. Each hidden neuron ends up doing several jobs at once, because there aren't enough neurons to give each useful feature its own dedicated neuron.

This is your first encounter with polysemanticity — one neuron firing for many unrelated features. It is the single biggest obstacle in modern mech interp, and it's why the next project (`01-toy-models-superposition/`) exists.

## 7. Discussion: what you just did

You just performed the most foundational move in mechanistic interpretability:

1. Trained a model.
2. Opened it up and looked at the weights.
3. Found that the weights encode something a human can recognise.
4. Did the same thing on a slightly bigger model and saw the picture get messier.

Every subsequent project in this curriculum is a variation on this same move:

- Project 1 (superposition): why are the MLP filters messy? Because features are packed into fewer dimensions than there are features. We'll see this happen in the cleanest possible toy setting.
- Project 2 (grokking): models don't only learn templates — they can learn full algorithms (like modular arithmetic via Fourier series). And generalisation can happen in sudden, late jumps that contradict your intuitions about training.
- Project 3 (induction heads): in transformers trained on natural text, circuits of attention heads cooperate to implement specific algorithms — like "copy from the last occurrence of this token."
- Project 4 (IOI): in pre-trained GPT-2, you can find and causally verify circuits using activation patching.
- Project 5 (SAEs): you can train an auxiliary model to automatically recover monosemantic features from a model's hidden states — closing the loop with project 1's superposition problem.

Onwards to project 1.